# Question 16 - 42. Trapping Rain Water

Given `n` non-negative integers representing an elevation map where the width of each bar is `1`, compute how much water it can trap after raining.

**Example 1:**

    Input: height = [0,1,0,2,1,0,1,3,2,1,2,1]
    Output: 6
    Explanation: The above elevation map is represented by array [0,1,0,2,1,0,1,3,2,1,2,1]. In this case, 6 units of rain water are being trapped.

                  █
          █ ≈ ≈ ≈ █ █ ≈ █
      █ ≈ █ █ ≈ █ █ █ █ █ █
    ───────────────────────
    0 1 0 2 1 0 1 3 2 1 2 1      (█ = bar, ≈ = trapped water)

**Example 2:**

    Input: height = [4,2,0,3,2,5]
    Output: 9

**Constraints:**

- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Prefix max / suffix max, then two pointers

📘 **Revise first:** [Pattern D · prefix & suffix](./0-concepts-array-and-string.ipynb) · [Pattern B · two pointers](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (scan both sides per bar) | O(n²) | O(1) |
| 1½ Precomputed left-max / right-max arrays | O(n) | O(n) |
| 2️⃣ Optimized (two pointers) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

The numbers are the heights of walls (bars), each 1 unit wide. Rain falls and water collects in the dips between taller walls. How many units of water are trapped?

```
heights: [4, 2, 0, 3, 2, 5]

  5 |           █
  4 | █ ≈ ≈ ≈ ≈ █
  3 | █ ≈ ≈ █ ≈ █
  2 | █ █ ≈ █ █ █
  1 | █ █ ≈ █ █ █
    └─────────────
      4 2 0 3 2 5        → 9 units of water (≈)
```

**Think about ONE bar at a time.** How high does water stand above bar `i`? Water rises until it would spill over the **lower** of the two "walls" around it:
- the **tallest bar anywhere to its left**, and
- the **tallest bar anywhere to its right**.

> **water at i = min(tallest on the left, tallest on the right) − height[i]** (never below 0)

**Everyday picture: a bathtub with two walls of different heights.** Fill it and the water level is set by the **shorter** wall. Water spills over the short side however tall the other wall is. The answer is just this rule added up over every bar.

### Step 0 · Clarify before coding

🗣️ *"Bars are width 1, heights are non-negative. I'll compute the water standing above each bar and add them up. Water above a bar is the smaller of the tallest bar on its left and the tallest on its right, minus its own height."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Does water spill off the ends? | **Yes.** | The first and last bars never hold water. |
| Heights non-negative integers? | **Yes.** | |
| Fewer than 3 bars? | Always 0. | |
| Size? | Up to 2·10⁴. | O(n²) ≈ 4·10⁸, too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** apply the formula literally. For each bar, **scan left** to find the tallest bar there, **scan right** for the tallest there, and add `min(left, right) − height`.

🗣️ *"Directly from the formula: for every index, scan both sides for the tallest wall, take the smaller, and subtract the bar's height. That's O(n) per bar, so O(n²) total."*

**The waste:** neighbouring bars scan almost the same stretches. Bar 5's "tallest on the left" is just bar 4's answer compared with bar 4 itself, but we rescan from scratch.

In [1]:
class SolutionBrute:
    def trap(self, height: list[int]) -> int:
        total = 0
        for i in range(len(height)):
            tallest_left = max(height[: i + 1])     # includes the bar itself
            tallest_right = max(height[i:])
            total += min(tallest_left, tallest_right) - height[i]
        return total

#### Step 1½ · Precompute the tallest-so-far from each side

"Tallest bar from the start up to `i`" is a **running maximum**: `left_max[i] = max(left_max[i−1], height[i])`. One pass left to right builds it for every bar. Do the same from the right. Then each bar's water is O(1).

🗣️ *"I'll precompute the running maximum from the left and from the right in two passes, then one pass to add up the water. That's O(n) time but uses two extra arrays. Can I avoid storing them?"*

In [2]:
class SolutionPrefix:
    def trap(self, height: list[int]) -> int:
        n = len(height)
        left_max, right_max = [0] * n, [0] * n
        left_max[0] = height[0]
        for i in range(1, n):
            left_max[i] = max(left_max[i - 1], height[i])
        right_max[-1] = height[-1]
        for i in range(n - 2, -1, -1):
            right_max[i] = max(right_max[i + 1], height[i])
        return sum(min(left_max[i], right_max[i]) - height[i] for i in range(n))

### Step 2 · Optimize: the "aha" moment

We store two whole arrays, but for each bar we only ever use the **smaller** of its two values. Do we always need to know both exactly?

> 💡 **Aha:** put one pointer at each end. Keep `left_max` (tallest seen from the left so far) and `right_max` (tallest seen from the right so far).
> If `left_max < right_max`, then for the **left pointer's** bar, the true tallest-on-the-right is **at least** `right_max`. The unexplored middle can only make it taller, never shorter. So the smaller wall is definitely `left_max`, and we already know it.
> → Water at the left bar = `left_max − height[left]`. Count it and move the left pointer in.
> Otherwise, the same logic settles the **right** bar.

**Rule: always move the side whose max is smaller**, because that side's water is already decided.

🗣️ *"The water at a bar is decided by its shorter wall. With two pointers from both ends, I track the tallest bar seen from each side. If the left side's tallest is smaller, then for the left pointer's bar the left wall is the limiting one, because whatever's in the middle can only make the right wall taller. So I add left_max minus its height and move left. Otherwise I do the mirror image on the right. One pass, constant space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Scan both sides for every bar | ❌ | O(n²). Recomputes the same maximums. |
| Left-max / right-max arrays | ✅ clearest | O(n). Easiest to prove; costs two O(n) arrays. Say this first. |
| Monotonic **stack** (fill water layer by layer) | ✅ alternative | O(n) time, O(n) stack. Fills each basin when a taller bar closes it. Good to know, harder to get right live. |
| **Two pointers + two running maxes** | ✅ **best** | O(n) time and **O(1)** memory, using "the shorter wall decides". |
| Segment tree for range maximums | ❌ | Overkill. We only need "from the start" and "from the end" maximums, not arbitrary ranges. |

**Interview strategy:** state the per-bar formula → show the two-array version (proves you understand it) → upgrade to two pointers.

In [3]:
class Solution:
    def trap(self, height: list[int]) -> int:
        left, right = 0, len(height) - 1
        left_max = right_max = 0
        total = 0
        while left < right:
            left_max = max(left_max, height[left])
            right_max = max(right_max, height[right])
            if left_max < right_max:
                total += left_max - height[left]     # left wall is the limit here
                left += 1
            else:
                total += right_max - height[right]   # right wall is the limit here
                right -= 1
        return total

### Step 3 · Toy example walkthrough (tell it like a story)

`height = [4, 2, 0, 3, 2, 5]` (expected 9)

| step | left (h) | right (h) | left_max | right_max | smaller side | water added | total |
|---|---|---|---|---|---|---|---|
| 1 | 0 (4) | 5 (5) | 4 | 5 | left | 4 − 4 = 0 | 0 |
| 2 | 1 (2) | 5 (5) | 4 | 5 | left | 4 − 2 = **2** | 2 |
| 3 | 2 (0) | 5 (5) | 4 | 5 | left | 4 − 0 = **4** | 6 |
| 4 | 3 (3) | 5 (5) | 4 | 5 | left | 4 − 3 = **1** | 7 |
| 5 | 4 (2) | 5 (5) | 4 | 5 | left | 4 − 2 = **2** | 9 |
| stop | 5 | 5 | | | pointers met | | **9** ✅ |

**Narrated:** "The left wall is 4 and the right wall is 5. Since 4 is shorter, the water level for everything I settle from the left is capped at 4: the right side is guaranteed to be at least 5. So bar 1 holds 2, bar 2 holds 4, bar 3 holds 1, bar 4 holds 2. Total 9."

**Check with the formula:** index 1 → min(4, 5) − 2 = 2. Index 2 → 4. Index 3 → 1. Index 4 → 2. Sum = 9 ✅

In [4]:
cases = [
    ([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], 6),
    ([4, 2, 0, 3, 2, 5], 9),
    ([1], 0),
    ([2, 0], 0),
    ([3, 0, 3], 3),
    ([1, 2, 3, 4], 0),            # only rising -> nothing trapped
    ([5, 4, 1, 2], 1),
]
for h, expected in cases:
    for Impl in (SolutionBrute, SolutionPrefix, Solution):
        assert Impl().trap(h) == expected, Impl.__name__
    print(h, "->", expected)

import random
for _ in range(500):
    h = [random.randint(0, 6) for _ in range(random.randint(1, 12))]
    assert Solution().trap(h) == SolutionBrute().trap(h), h
print("All tests passed ✅")

[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] -> 6
[4, 2, 0, 3, 2, 5] -> 9
[1] -> 0
[2, 0] -> 0
[3, 0, 3] -> 3
[1, 2, 3, 4] -> 0
[5, 4, 1, 2] -> 1
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Scan both sides per bar | O(n²) | O(1) | For every bar, look at every other bar. |
| Left/right max arrays | O(n) | O(n) | Three passes, two helper arrays. |
| Monotonic stack | O(n) | O(n) | Each bar pushed and popped once. |
| **Two pointers** | **O(n)** | **O(1)** | Each step settles one bar for good. |

**Why O(n) time?** Every loop step moves one pointer one step inward, and they stop when they meet. So there are about n steps, each constant work.

**Why O(1) memory?** Five numbers: two pointers, two maxes, and the total.

**Edge cases to mention:** fewer than 3 bars (0), only rising or only falling (0), all zeros (0), two equal tall walls (`[3,0,3]` → 3).

**Likely follow-ups:**
- *A 2D height map* (LC 407) → use a min-heap working inward from the border. The same "lowest wall decides" idea.
- *Don't confuse it with Container With Most Water* (Q27) → there you pick two lines and ignore the bars in between.

---

#### 🗣️ Full interview script (about 60 seconds)

*"The water above any bar equals the smaller of the tallest bar to its left and the tallest bar to its right, minus the bar's own height. I just need to add that up.*

*Brute force scans both sides for every bar, which is O(n²). Precomputing a running max from the left and from the right gives O(n) time with two O(n) arrays.*

*To get O(1) space, I use two pointers from the ends and track the tallest bar seen from each side. If the left max is smaller than the right max, the left pointer's bar is limited by the left max, because the unexplored middle can only make the right side taller. So I add left_max minus its height and move left. Otherwise I do the mirror image on the right.*

*Each step settles one bar: O(n) time, O(1) space."*